# EPA Hyperparameter Tuning v3 -- Representation-Reading Based

Uses **representation reading** (RepReadingPipeline + rep_readers) to score
steering quality, instead of Likert log-probabilities.

**Rationale:** The pipeline projects hidden states onto extracted EPA direction
vectors for each layer. A single forward pass returns scores for all layers at once.
These per-layer scores are used directly as the regression target, giving a much
faster and more internally-consistent tuning signal.

**Metric:** Spearman rank correlation between per-layer direction-projection scores
and continuous EPA dictionary values (not binned Likert labels).

**Pipeline:**
1. Baseline: run read pipeline on all utterances, collect per-layer projection scores.
2. Phase 1: for each (layer, coeff, dimension), apply steering then re-read that layer.
3. Simple Independent: combine all Phase-1-effective layers with their best coefficients.
4. Phase 2: greedy forward selection from top-K Phase-1 layers.
5. Final validation: evaluate full dataset with cross-dimension interference.

**Prerequisites:** `epa_tuning_dataset.json` from `hyperparameter_tuning.ipynb` Cell 3.

In [ ]:
\
# === Cell 1: Setup & Imports ===
import sys; sys.path.append('../..')
import json, pickle
from datetime import datetime
import torch, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from repe import repe_pipeline_registry
from repe.rep_control_reading_vec import WrappedReadingVecModel
repe_pipeline_registry()

from examples.act_new.utils import (
    format_for_reading,
    make_epa_activations,
    read_epa_scores,
)
print("All imports OK.")

In [ ]:
\
# === Cell 2: Load Model, Directions & Reading Pipeline ===
with open("epa_directions.pkl", 'rb') as f:
    directions_data = pickle.load(f)
rep_readers = directions_data['rep_readers']
hidden_layers = directions_data['hidden_layers']
model_name = directions_data['model_name']
print(f"Loaded directions: {model_name}")
print(f"  {len(hidden_layers)} layers: {hidden_layers[0]}..{hidden_layers[-1]}")

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

# RepReadingPipeline: one forward pass reads all layers at once
rep_pipeline = pipeline(
    "rep-reading",
    model=model,
    tokenizer=tokenizer,
)

# Layer index helpers
n_layers = len(model.model.layers)
def neg_to_pos(x):
    if isinstance(x, (list, tuple)):
        return [n_layers + k for k in x]
    return n_layers + x

# WrappedReadingVecModel for steering
wrapped_model = WrappedReadingVecModel(model, tokenizer)

DIMENSIONS = ["evaluation", "potency", "activity"]
INTERFERENCE_PENALTY = 0.5
print(f"Model loaded ({n_layers} layers). Rep pipeline ready.")

In [ ]:
\
# === Cell 3: Load Dataset ===
with open("epa_tuning_dataset.json", 'r') as f:
    dataset = json.load(f)
utterances = dataset['utterances']
print(f"Loaded {len(utterances)} utterances")

# Ground truth: continuous EPA values (not binned Likert labels)
# Shape: ground_truth[dim] = list of floats (raw dictionary EPA values)
ground_truth = {
    "evaluation": [u['target_epa']['e'] for u in utterances],
    "potency":    [u['target_epa']['p'] for u in utterances],
    "activity":   [u['target_epa']['a'] for u in utterances],
}

# Format each utterance for reading (assistant-position format)
formatted_texts = [format_for_reading(u['text']) for u in utterances]

print("\nGround-truth EPA statistics:")
for dim in DIMENSIONS:
    vals = ground_truth[dim]
    print(f"  {dim:12s}: min={min(vals):+.2f} max={max(vals):+.2f} "
          f"mean={np.mean(vals):+.2f} std={np.std(vals):.2f}")

In [ ]:
\
# === Cell 4: Representation-Reading Utility ===
# The RepReadingPipeline returns per-layer projection scores in one pass.
# We extract the score for each layer individually to use as our tuning signal.

def read_per_layer_scores(text, dimension):
    # Run the reading pipeline and return a dict of {layer: score}.
    # All layers from the stored rep_reader are scored in one forward pass.
    reader = rep_readers[dimension]
    all_layers = list(reader.directions.keys())  # negative indices
    result = rep_pipeline(
        [format_for_reading(text)],
        hidden_layers=all_layers,
        rep_reader=reader,
        padding=True,
        truncation=True,
    )
    # result[0] is a dict: {layer: score_value}
    return {layer: result[0][layer] for layer in all_layers if layer in result[0]}


def batch_read_scores(texts, dimension):
    # Run the reading pipeline on a list of texts for one dimension.
    # Returns a dict {layer: [score_per_text]}.
    reader = rep_readers[dimension]
    all_layers = list(reader.directions.keys())
    results = rep_pipeline(
        [format_for_reading(t) for t in texts],
        hidden_layers=all_layers,
        rep_reader=reader,
        padding=True,
        truncation=True,
        batch_size=8,
    )
    # results is a list of dicts (one per text), each {layer: score}
    layer_scores = {layer: [] for layer in all_layers}
    for r in results:
        for layer in all_layers:
            layer_scores[layer].append(r.get(layer, 0.0))
    return layer_scores


# --- Quick sanity check ---
print("Sanity check: reading first utterance...")
for dim in DIMENSIONS:
    scores = read_per_layer_scores(utterances[0]['text'], dim)
    layers_sorted = sorted(scores.keys())
    mid = layers_sorted[len(layers_sorted)//2]
    print(f"  {dim:12s}: layer {mid} score = {scores[mid]:.4f}")
print("OK.")

In [ ]:
\
# === Cell 5: Baseline Per-Layer Scores (No Steering) ===
# Run the reading pipeline once on all utterances and collect per-layer scores.
# This is fast: one forward pass per utterance extracts all layers at once.

print("Computing baseline per-layer scores for all utterances...")
texts = [u['text'] for u in utterances]

baseline_scores = {}  # {dim: {layer: [score_per_utt]}}
for dim in tqdm(DIMENSIONS, desc="Dimensions"):
    baseline_scores[dim] = batch_read_scores(texts, dim)

# Compute per-layer Spearman correlation with ground truth (no steering)
baseline_rho = {}  # {dim: {layer: rho}}
print("\n=== Baseline Per-Layer Spearman (no steering) ===")
for dim in DIMENSIONS:
    baseline_rho[dim] = {}
    layer_rhos = []
    for layer, scores in baseline_scores[dim].items():
        rho, _ = spearmanr(scores, ground_truth[dim])
        baseline_rho[dim][layer] = float(rho)
        layer_rhos.append((layer, rho))

    layer_rhos.sort(key=lambda x: abs(x[1]), reverse=True)
    print(f"\n{dim.upper()} -- top 5 baseline layers:")
    for layer, rho in layer_rhos[:5]:
        print(f"  layer {layer:4d} (pos {neg_to_pos(layer):2d}): rho={rho:+.4f}")

# Build evaluation subset for Phase 1/2 speed
N_SWEEP = 40
np.random.seed(42)
sweep_idx = sorted(np.random.choice(len(utterances), min(N_SWEEP, len(utterances)), replace=False))
sweep_texts = [texts[i] for i in sweep_idx]
sweep_gt = {d: [ground_truth[d][i] for i in sweep_idx] for d in DIMENSIONS}
sweep_baseline = {d: {l: [baseline_scores[d][l][i] for i in sweep_idx]
                       for l in baseline_scores[d]}
                  for d in DIMENSIONS}
print(f"\nSweep subset: {len(sweep_idx)}/{len(utterances)} utterances for Phase 1 & 2")

In [ ]:
\
# === Cell 6: Baseline Per-Layer Correlation Visualization ===
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for idx, dim in enumerate(DIMENSIONS):
    layers_sorted = sorted(baseline_rho[dim].keys())
    rhos = [baseline_rho[dim][l] for l in layers_sorted]
    pos_idx = [neg_to_pos(l) for l in layers_sorted]
    colors = ['#2196F3' if r >= 0 else '#F44336' for r in rhos]
    axes[idx].bar(range(len(layers_sorted)), rhos, color=colors)
    axes[idx].set_xticks(range(0, len(layers_sorted), 3))
    axes[idx].set_xticklabels([str(pos_idx[i]) for i in range(0, len(layers_sorted), 3)],
                               rotation=45, fontsize=8)
    axes[idx].set_xlabel("Layer (positive index)")
    axes[idx].set_ylabel("Spearman rho")
    axes[idx].set_title(f"{dim.capitalize()} -- Baseline Per-Layer Correlation")
    axes[idx].axhline(0, color='gray', ls='--', alpha=0.5)
    # Annotate best layer
    best_l = max(baseline_rho[dim], key=lambda l: abs(baseline_rho[dim][l]))
    best_rho = baseline_rho[dim][best_l]
    axes[idx].annotate(f"best: {neg_to_pos(best_l)}\nrho={best_rho:+.3f}",
        xy=(layers_sorted.index(best_l), best_rho),
        xytext=(5, 5), textcoords='offset points', fontsize=8)
plt.suptitle("Baseline: Reading Direction Projection vs Target EPA", fontsize=13)
plt.tight_layout()
plt.savefig("v3_baseline.png", dpi=150, bbox_inches='tight')
plt.show()
print("Baseline visualization saved.")

In [ ]:
\
# === Cell 7: Phase 1 -- Individual Layer Sweep ===
# For each (layer, coeff, dimension):
#   1. Apply single-layer steering at the given coeff
#   2. Run reading pipeline to get projection scores for that layer
#   3. Compute Spearman rho vs target EPA values
#
# Speed: ~5 min total (40 utts x 31 layers x 8 coeffs x 3 dims)
# Faster than Likert because reading pipeline does one forward pass.

COEFF_CANDIDATES = [0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.75, 1.0]
phase1 = {dim: {} for dim in DIMENSIONS}

total = len(hidden_layers) * len(COEFF_CANDIDATES) * len(DIMENSIONS)
print(f"Phase 1: {total} combos ({len(hidden_layers)} layers x {len(COEFF_CANDIDATES)} "
      f"coeffs x {len(DIMENSIONS)} dims)")
print(f"Using {len(sweep_idx)}-utterance subset for speed.\n")

pbar = tqdm(total=total, desc="Phase 1")

for steer_dim in DIMENSIONS:
    for neg_layer in hidden_layers:
        pos_layer = neg_to_pos(neg_layer)
        coeff_data = {}

        for coeff in COEFF_CANDIDATES:
            # Build single-layer activation
            kwargs = {f"{d[0]}_coeff": 0.0 for d in DIMENSIONS}
            kwargs[f"{steer_dim[0]}_coeff"] = coeff
            act = make_epa_activations(
                rep_readers=rep_readers, layers=[neg_layer],
                device=model.device, dtype=model.dtype,
                normalize=True, **kwargs)

            # Apply steering
            wrapped_model.unwrap()
            wrapped_model.wrap_block([pos_layer], block_name="decoder_block")
            wrapped_model.set_controller([pos_layer], {pos_layer: act[neg_layer]}, "decoder_block")

            # Read all three dimensions (to measure interference later)
            steered_scores = {d: [] for d in DIMENSIONS}
            for t in sweep_texts:
                for d in DIMENSIONS:
                    reader = rep_readers[d]
                    all_layers = list(reader.directions.keys())
                    result = rep_pipeline(
                        [format_for_reading(t)],
                        hidden_layers=[neg_layer],  # only read the steered layer
                        rep_reader=reader,
                        padding=True, truncation=True,
                    )
                    steered_scores[d].append(result[0].get(neg_layer, 0.0))

            wrapped_model.reset(); wrapped_model.unwrap()

            # Compute Spearman for each dimension
            rhos = {}
            deltas = {}
            for d in DIMENSIONS:
                rho, _ = spearmanr(steered_scores[d], sweep_gt[d])
                base_rho = baseline_rho[d][neg_layer]
                rhos[d] = float(rho)
                deltas[d] = float(rho - base_rho)

            # Composite score: on-target delta minus interference penalty
            others = [d for d in DIMENSIONS if d != steer_dim]
            score = deltas[steer_dim] - INTERFERENCE_PENALTY * np.mean([abs(deltas[d]) for d in others])
            coeff_data[coeff] = {"rhos": rhos, "deltas": deltas, "score": float(score)}
            pbar.update(1)

        # Best coeff = highest composite score
        best_c = max(coeff_data, key=lambda c: coeff_data[c]["score"])
        phase1[steer_dim][neg_layer] = {
            "coeff_data": coeff_data,
            "best_coeff": best_c,
            "best_score": coeff_data[best_c]["score"],
            "best_delta": coeff_data[best_c]["deltas"][steer_dim],
            "best_rho": coeff_data[best_c]["rhos"][steer_dim],
        }

pbar.close()
print("\nPhase 1 complete! Top 10 layers per dimension:")
for dim in DIMENSIONS:
    ranked = sorted(phase1[dim].items(), key=lambda x: x[1]['best_score'], reverse=True)
    print(f"\n{dim.upper()}:")
    for layer, info in ranked[:10]:
        print(f"  layer {layer:4d} (pos {neg_to_pos(layer):2d}): "
              f"rho={info['best_rho']:+.3f} delta={info['best_delta']:+.3f} "
              f"score={info['best_score']:+.3f} coeff={info['best_coeff']}")

In [ ]:
\
# === Cell 8: Phase 1 Visualization ===
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
for idx, dim in enumerate(DIMENSIONS):
    sorted_layers = sorted(hidden_layers)
    matrix = np.zeros((len(sorted_layers), len(COEFF_CANDIDATES)))
    for i, layer in enumerate(sorted_layers):
        for j, coeff in enumerate(COEFF_CANDIDATES):
            matrix[i, j] = phase1[dim][layer]['coeff_data'][coeff]['score']
    sns.heatmap(matrix, cmap="RdBu_r", center=0, vmin=-0.3, vmax=0.3,
        xticklabels=[f"{c}" for c in COEFF_CANDIDATES],
        yticklabels=[f"{neg_to_pos(l)}" for l in sorted_layers],
        ax=axes[idx])
    axes[idx].set_xlabel("Coefficient"); axes[idx].set_ylabel("Layer (pos)")
    axes[idx].set_title(f"{dim.capitalize()} -- Composite Score")
plt.suptitle("Phase 1: Per-Layer Composite Score (delta_target - penalty*interference)", fontsize=12)
plt.tight_layout()
plt.savefig("v3_phase1_heatmap.png", dpi=150, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, dim in enumerate(DIMENSIONS):
    ranked = sorted(phase1[dim].items(), key=lambda x: x[1]['best_score'], reverse=True)[:15]
    labels = [f"{neg_to_pos(l)}" for l, _ in ranked]
    scores = [info['best_score'] for _, info in ranked]
    deltas = [info['best_delta'] for _, info in ranked]
    rhos = [info['best_rho'] for _, info in ranked]
    coeffs = [info['best_coeff'] for _, info in ranked]
    bars = axes[idx].barh(range(len(labels)), scores,
                          color=plt.cm.viridis(np.array(coeffs) / max(COEFF_CANDIDATES)))
    axes[idx].set_yticks(range(len(labels))); axes[idx].set_yticklabels(labels)
    axes[idx].invert_yaxis(); axes[idx].set_xlabel("Composite Score")
    axes[idx].axvline(0, color='gray', ls='--', alpha=0.5)
    axes[idx].set_title(f"{dim.capitalize()} -- Top 15 Layers")
    for i, (s, c, d) in enumerate(zip(scores, coeffs, deltas)):
        axes[idx].annotate(f"c={c} d={d:+.2f}", xy=(s, i), fontsize=7, va='center')
plt.suptitle("Phase 1: Top Layers (color=best coefficient)", fontsize=13)
plt.tight_layout()
plt.savefig("v3_phase1_top.png", dpi=150, bbox_inches='tight'); plt.show()
print("Phase 1 visualizations saved.")

In [ ]:
\
# === Cell 9: Simple Independent Combination ===
# Combines all Phase-1-effective layers with their best coefficients.
# Assumes layer effects are additive (independence assumption).
# Validated on full dataset.

MIN_SCORE = 0.01
simple_results = {}

for steer_dim in DIMENSIONS:
    # Select layers with positive composite score
    selected = {l: info['best_coeff']
                for l, info in phase1[steer_dim].items()
                if info['best_score'] > MIN_SCORE}
    if not selected:
        best_l = max(phase1[steer_dim].items(), key=lambda x: x[1]['best_score'])
        selected = {best_l[0]: best_l[1]['best_coeff']}

    print(f"\n{'='*60}\nSimple {steer_dim.upper()}: {len(selected)} layers")
    neg_layers = list(selected.keys())
    pos_layers = neg_to_pos(neg_layers)

    # Build combined per-layer activations (different coeff per layer)
    combined = {}
    for nl, c in selected.items():
        kw = {f"{d[0]}_coeff": 0.0 for d in DIMENSIONS}
        kw[f"{steer_dim[0]}_coeff"] = c
        a = make_epa_activations(rep_readers=rep_readers, layers=[nl],
            device=model.device, dtype=model.dtype, normalize=True, **kw)
        combined[neg_to_pos(nl)] = a[nl]

    wrapped_model.unwrap()
    wrapped_model.wrap_block(pos_layers, block_name="decoder_block")
    wrapped_model.set_controller(pos_layers, combined, "decoder_block")

    # Read all dimensions via pipeline (full dataset)
    # For multi-layer, use all neg_layers as reading target, average them
    all_dim_scores = {d: [] for d in DIMENSIONS}
    for t in tqdm(texts, desc=f"Simple {steer_dim[:4]}", leave=False):
        for d in DIMENSIONS:
            result = rep_pipeline(
                [format_for_reading(t)], hidden_layers=neg_layers,
                rep_reader=rep_readers[d], padding=True, truncation=True)
            layer_vals = [result[0].get(l, 0.0) for l in neg_layers]
            all_dim_scores[d].append(float(np.mean(layer_vals)))

    wrapped_model.reset(); wrapped_model.unwrap()

    corrs = {d: float(spearmanr(all_dim_scores[d], ground_truth[d])[0]) for d in DIMENSIONS}
    base_corrs = {d: float(np.mean([baseline_rho[d][l] for l in selected])) for d in DIMENSIONS}
    delts = {d: corrs[d] - base_corrs[d] for d in DIMENSIONS}
    others = [d for d in DIMENSIONS if d != steer_dim]
    score = corrs[steer_dim] - INTERFERENCE_PENALTY * np.mean([abs(delts[d]) for d in others])
    simple_results[steer_dim] = {
        "selected": selected, "correlations": corrs, "deltas": delts, "score": score}

    print(f"  On-target: rho={corrs[steer_dim]:+.3f} delta={delts[steer_dim]:+.3f} score={score:.3f}")
    for d in others:
        print(f"  {d:12s}: rho={corrs[d]:+.3f} delta={delts[d]:+.3f}")
print("\nSimple independent done.")

In [ ]:
\
# === Cell 10: Phase 2 -- Greedy Forward Selection ===
# Starting from the best Phase-1 layer, greedily adds layers from top-K candidates.
# Each candidate is tested with all coefficient values; best is kept.
# Uses sweep subset (40 utterances) for speed; validated on full set in Cell 11.

TOP_K = 10; MAX_LAYERS = 8; MIN_IMPROVEMENT = 0.002

def eval_combo(selected_lc, steer_dim):
    # Evaluate on sweep subset. Returns composite score + per-dim rhos.
    nls = list(selected_lc.keys())
    pls = neg_to_pos(nls)
    comb = {}
    for nl, c in selected_lc.items():
        kw = {f"{d[0]}_coeff": 0.0 for d in DIMENSIONS}
        kw[f"{steer_dim[0]}_coeff"] = c
        a = make_epa_activations(rep_readers=rep_readers, layers=[nl],
            device=model.device, dtype=model.dtype, normalize=True, **kw)
        comb[neg_to_pos(nl)] = a[nl]
    wrapped_model.unwrap()
    wrapped_model.wrap_block(pls, block_name="decoder_block")
    wrapped_model.set_controller(pls, comb, "decoder_block")
    steered = {d: [] for d in DIMENSIONS}
    for t in sweep_texts:
        for d in DIMENSIONS:
            result = rep_pipeline(
                [format_for_reading(t)], hidden_layers=nls,
                rep_reader=rep_readers[d], padding=True, truncation=True)
            layer_vals = [result[0].get(l, 0.0) for l in nls]
            steered[d].append(float(np.mean(layer_vals)))
    wrapped_model.reset(); wrapped_model.unwrap()
    corrs = {}; delts = {}
    for d in DIMENSIONS:
        rho, _ = spearmanr(steered[d], sweep_gt[d])
        base_rho = float(np.mean([baseline_rho[d][l] for l in nls]))
        corrs[d] = float(rho); delts[d] = float(rho - base_rho)
    others = [d for d in DIMENSIONS if d != steer_dim]
    score = corrs[steer_dim] - INTERFERENCE_PENALTY * np.mean([abs(delts[d]) for d in others])
    return {"correlations": corrs, "deltas": delts, "score": float(score)}

print(f"Phase 2: Greedy (top-{TOP_K} from Phase 1, max {MAX_LAYERS} layers, "
      f"min improvement {MIN_IMPROVEMENT})")
greedy_results = {}

for steer_dim in DIMENSIONS:
    print(f"\n{'='*60}\nGreedy: {steer_dim.upper()}\n{'='*60}")
    ranked = sorted(phase1[steer_dim].items(),
                    key=lambda x: x[1]['best_score'], reverse=True)[:TOP_K]
    candidates = [l for l, _ in ranked]
    best_l, best_info = ranked[0]
    selected = {best_l: best_info['best_coeff']}
    ev = eval_combo(selected, steer_dim)
    cur_score = ev['score']
    print(f"Step 0: layer {best_l} coeff={best_info['best_coeff']} "
          f"rho={ev['correlations'][steer_dim]:+.3f} score={cur_score:.4f}")
    history = [{"step": 0, "selected": dict(selected), "score": cur_score, **ev}]

    for step in range(1, MAX_LAYERS):
        remaining = [l for l in candidates if l not in selected]
        if not remaining:
            break
        best_add_score = cur_score
        best_add_layer = None; best_add_coeff = None
        for cand_layer in remaining:
            for coeff in COEFF_CANDIDATES:
                trial = {**selected, cand_layer: coeff}
                ev = eval_combo(trial, steer_dim)
                if ev['score'] > best_add_score:
                    best_add_score = ev['score']
                    best_add_layer = cand_layer; best_add_coeff = coeff
        if best_add_layer is None or (best_add_score - cur_score) < MIN_IMPROVEMENT:
            print(f"Step {step}: no improvement (best gain={(best_add_score - cur_score):.4f}), stopping.")
            break
        selected[best_add_layer] = best_add_coeff
        cur_score = best_add_score
        ev = eval_combo(selected, steer_dim)
        print(f"Step {step}: +layer {best_add_layer} (pos {neg_to_pos(best_add_layer)}) "
              f"coeff={best_add_coeff} rho={ev['correlations'][steer_dim]:+.3f} "
              f"score={cur_score:.4f}")
        history.append({"step": step, "selected": dict(selected), "score": cur_score, **ev})

    greedy_results[steer_dim] = {"selected": dict(selected), "history": history, "score": cur_score}
    print(f"\nFinal: {len(selected)} layers, score={cur_score:.4f}")
    for l, c in sorted(selected.items()):
        print(f"  layer {l:4d} (pos {neg_to_pos(l):2d}): coeff={c}")

print("\nPhase 2 complete!")

In [ ]:
\
# === Cell 11: Final Evaluation on Full Dataset ===
final_results = {}

for steer_dim in DIMENSIONS:
    sel = greedy_results[steer_dim]['selected']
    nls = list(sel.keys()); pls = neg_to_pos(nls)
    comb = {}
    for nl, c in sel.items():
        kw = {f"{d[0]}_coeff": 0.0 for d in DIMENSIONS}
        kw[f"{steer_dim[0]}_coeff"] = c
        a = make_epa_activations(rep_readers=rep_readers, layers=[nl],
            device=model.device, dtype=model.dtype, normalize=True, **kw)
        comb[neg_to_pos(nl)] = a[nl]
    wrapped_model.unwrap()
    wrapped_model.wrap_block(pls, block_name="decoder_block")
    wrapped_model.set_controller(pls, comb, "decoder_block")
    steered = {d: [] for d in DIMENSIONS}
    for t in tqdm(texts, desc=f"Final {steer_dim[:4]}", leave=False):
        for d in DIMENSIONS:
            result = rep_pipeline(
                [format_for_reading(t)], hidden_layers=nls,
                rep_reader=rep_readers[d], padding=True, truncation=True)
            layer_vals = [result[0].get(l, 0.0) for l in nls]
            steered[d].append(float(np.mean(layer_vals)))
    wrapped_model.reset(); wrapped_model.unwrap()
    corrs = {d: float(spearmanr(steered[d], ground_truth[d])[0]) for d in DIMENSIONS}
    base_corrs = {d: float(np.mean([baseline_rho[d][l] for l in nls])) for d in DIMENSIONS}
    delts = {d: corrs[d] - base_corrs[d] for d in DIMENSIONS}
    final_results[steer_dim] = {
        "selected": sel, "correlations": corrs, "deltas": delts,
        "steered_scores": {d: steered[d] for d in DIMENSIONS}
    }

print("\n=== Final Results (full dataset) ===")
for sd in DIMENSIONS:
    r = final_results[sd]
    print(f"\n{sd.upper()} ({len(r['selected'])} layers, "
          f"rho={r['correlations'][sd]:+.3f} delta={r['deltas'][sd]:+.3f}):")
    for d in DIMENSIONS:
        if d != sd:
            print(f"  {d:12s}: rho={r['correlations'][d]:+.3f} delta={r['deltas'][d]:+.3f}")

In [ ]:
\
# === Cell 12: Final Visualization ===

# --- Interference matrix ---
interf = np.zeros((3, 3)); absol = np.zeros((3, 3))
for i, sd in enumerate(DIMENSIONS):
    for j, md in enumerate(DIMENSIONS):
        interf[i, j] = final_results[sd]['deltas'][md]
        absol[i, j] = final_results[sd]['correlations'][md]
dlabels = [d[:4].capitalize() for d in DIMENSIONS]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(interf, annot=True, fmt="+.3f", cmap="RdBu_r", center=0,
    xticklabels=dlabels, yticklabels=dlabels, ax=axes[0])
axes[0].set_xlabel("Measured"); axes[0].set_ylabel("Steered")
axes[0].set_title("Delta rho (diagonal=target)")
sns.heatmap(absol, annot=True, fmt=".3f", cmap="RdYlGn", vmin=-1, vmax=1,
    xticklabels=dlabels, yticklabels=dlabels, ax=axes[1])
axes[1].set_xlabel("Measured"); axes[1].set_ylabel("Steered")
axes[1].set_title("Absolute rho under best greedy steering")
plt.tight_layout()
plt.savefig("v3_interference.png", dpi=150, bbox_inches='tight'); plt.show()

# --- Scatter: steered read score vs target EPA ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, sd in enumerate(DIMENSIONS):
    sv = final_results[sd]['steered_scores'][sd]
    gv = ground_truth[sd]
    rho, _ = spearmanr(sv, gv)
    axes[idx].scatter(gv, sv, alpha=0.5, s=30, c='#2196F3')
    axes[idx].set_xlabel("Target EPA value"); axes[idx].set_ylabel("Steered read score")
    sel = greedy_results[sd]['selected']
    axes[idx].set_title(f"{sd.capitalize()}\n{len(sel)} layers, rho={rho:+.3f}")
plt.suptitle("Steered Read Score vs Target EPA (Greedy Selection)", fontsize=13)
plt.tight_layout()
plt.savefig("v3_scatter.png", dpi=150, bbox_inches='tight'); plt.show()

# --- Coefficient profiles ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, sd in enumerate(DIMENSIONS):
    sel = greedy_results[sd]['selected']
    ls = sorted(sel.keys())
    axes[idx].bar([str(neg_to_pos(l)) for l in ls], [sel[l] for l in ls], color='steelblue')
    axes[idx].set_xlabel("Layer (pos)"); axes[idx].set_ylabel("Coefficient")
    axes[idx].set_title(f"{sd.capitalize()} -- Per-Layer Coefficients")
    axes[idx].tick_params(axis='x', rotation=45)
plt.suptitle("Greedy-Selected Per-Layer Coefficient Profiles", fontsize=13)
plt.tight_layout()
plt.savefig("v3_coeff_profile.png", dpi=150, bbox_inches='tight'); plt.show()
print("Visualizations saved.")

In [ ]:
\
# === Cell 13: Save Results ===
output = {
    "metadata": {
        "model_name": model_name,
        "tuned_at": datetime.now().isoformat(),
        "method": "representation_reading",
        "metric": "spearman_rho_vs_target_epa",
        "n_utterances": len(utterances),
        "n_sweep": len(sweep_idx),
        "interference_penalty": INTERFERENCE_PENALTY,
    },
    "baseline_per_layer_rho": {
        d: {str(l): v for l, v in baseline_rho[d].items()} for d in DIMENSIONS
    },
    "coeff_candidates": COEFF_CANDIDATES,
    "simple_independent": {},
    "greedy_results": {},
}
for dim in DIMENSIONS:
    si = simple_results[dim]
    output["simple_independent"][dim] = {
        "selected_layers": {str(k): v for k, v in si['selected'].items()},
        "correlations": si['correlations'], "deltas": si['deltas'], "score": si['score'],
    }
    gr = greedy_results[dim]
    output["greedy_results"][dim] = {
        "selected_layers": {str(k): v for k, v in gr['selected'].items()},
        "n_layers": len(gr['selected']), "score": gr['score'],
        "history": [{"step": h["step"], "score": h["score"],
                     "selected": {str(k): v for k, v in h["selected"].items()}}
                    for h in gr['history']],
        "final_correlations": final_results[dim]['correlations'],
        "final_deltas": final_results[dim]['deltas'],
    }
with open("epa_tuning_v3_results.json", 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved to epa_tuning_v3_results.json")
print("\n" + "="*60 + "\nSUMMARY\n" + "="*60)
for dim in DIMENSIONS:
    gr = output['greedy_results'][dim]
    print(f"\n{dim.upper()}: {gr['n_layers']} layers, score={gr['score']:.4f}")
    for l, c in gr['selected_layers'].items():
        print(f"  Layer {l} (pos {neg_to_pos(int(l))}): coeff={c}")
    print(f"  rho(target): {gr['final_correlations'][dim]:+.3f} "
          f"(delta={gr['final_deltas'][dim]:+.3f})")